In [1]:
# ============================================================================
# IMPORT CÁC THƯ VIỆN CẦN THIẾT CHO XAI-RL FRAMEWORK
# ============================================================================

# 1. System & Path
import sys
import os
sys.path.append('d:\\NCKH\\SARSA_FinancialRL')

# 2. Data Processing
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 3. Deep Learning - PyTorch
import torch
from torch import nn
import torch.nn.functional as F

# 4. Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style cho plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 5. XAI Libraries - CHỈ DÙNG SHAP
try:
    import shap
    print("✓ SHAP available")
except ImportError:
    print("⚠ SHAP not installed - will install when needed")

# 6. Project-specific Imports
from agents.d_sarsa.d_sarsa import Qsa
from environments.stock_trading_env.mdp import StockTradingMDP
from data.data_provider.library_extracted.vnstock.VNStockDataProvider import VNStockDataProvider
from data.data_processor.feature_engineer import engineer_stat as es

# 7. Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("="*80)
print("✓ Tất cả thư viện đã được import thành công!")
print("="*80)
print("\n🎯 XAI-RL Framework - 3 Phương pháp độc lập:")
print("  [1] RDX  - Reward Decomposition (weights từ domain knowledge)")
print("  [2] MSX  - Multi-Step Explanation (trajectory analysis)")
print("  [3] SHAP - Feature Attribution (Shapley values)")
print("\n📊 Deep RL Agent:")
print("  • Qsa:              Q-network (input=7, output=11)")
print("  • StockTradingMDP:  Environment cho stock trading")
print("  • VNStockData:      Data provider cho VN market")
print("\nReady to analyze SARSA agent! 🚀")
print("="*80)

✓ SHAP available


✓ Tất cả thư viện đã được import thành công!

🎯 XAI-RL Framework - 3 Phương pháp độc lập:
  [1] RDX  - Reward Decomposition (weights từ domain knowledge)
  [2] MSX  - Multi-Step Explanation (trajectory analysis)
  [3] SHAP - Feature Attribution (Shapley values)

📊 Deep RL Agent:
  • Qsa:              Q-network (input=7, output=11)
  • StockTradingMDP:  Environment cho stock trading
  • VNStockData:      Data provider cho VN market

Ready to analyze SARSA agent! 🚀


In [2]:
# 1.1. Load model SARSA
qsa = Qsa(input_size=7, num_classes=11)
model_path_acb_phase_1 = 'd:\\NCKH\\SARSA_FinancialRL\\models\\sarsa_acb_phase_2.pth'
model_path_fpt_phase_1 = 'd:\\NCKH\\SARSA_FinancialRL\\models\\sarsa_fpt_phase_2.pth'
model_path_gas_phase_1 = 'd:\\NCKH\\SARSA_FinancialRL\\models\\sarsa_gas_phase_2.pth'
model_path_hpg_phase_1 = 'd:\\NCKH\\SARSA_FinancialRL\\models\\sarsa_hpg_phase_2.pth'
model_path_ssi_phase_1 = 'd:\\NCKH\\SARSA_FinancialRL\\models\\sarsa_ssi_phase_2.pth'
model_path_vcb_phase_1 = 'd:\\NCKH\\SARSA_FinancialRL\\models\\sarsa_vcb_phase_2.pth'

if os.path.exists(model_path_acb_phase_1):
    state_dict = torch.load(model_path_acb_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_acb_phase_1}")
else:
    print(f"✗ Model not found: {model_path_acb_phase_1}")

# # 1.2. Load dữ liệu 
# provider = VNStockDataProvider()
# print("\nĐang lấy dữ liệu từ vnstock...")
# df_raw = provider.get_ohlcv_data('SSI', '2021-12-14', '2024-12-31')
# print(f"✓ Đã lấy {len(df_raw)} dòng dữ liệu")

# # 1.3. Xử lý dữ liệu và thêm technical indicators
# df_processed = df_raw.copy()
# df_processed.rename(columns={'date': 'time'}, inplace=True)
# df_processed['time'] = pd.to_datetime(df_processed['time']).dt.strftime('%d/%m/%Y')
# df_processed = es.add_technical_indicators(df_processed, start_date= '01/01/2022')
# print(f"✓ Đã thêm technical indicators: {df_processed.shape}")

df_processed_test_ACB = pd.read_csv('D:\\NCKH\\SARSA_FinancialRL\\data\\data_storer\\data_research\\test\\test_ACB_phase_2.csv')
df_processed_test_ACB.rename(columns={'date': 'time'}, inplace=True)
df_processed_test_ACB['time'] = pd.to_datetime(df_processed_test_ACB['time']).dt.strftime('%d/%m/%Y')

df_processed_test_FPT = pd.read_csv('D:\\NCKH\\SARSA_FinancialRL\\data\\data_storer\\data_research\\test\\test_FPT_phase_2.csv')
df_processed_test_FPT.rename(columns={'date': 'time'}, inplace=True)
df_processed_test_FPT['time'] = pd.to_datetime(df_processed_test_FPT['time']).dt.strftime('%d/%m/%Y')
    
df_processed_test_GAS = pd.read_csv('D:\\NCKH\\SARSA_FinancialRL\\data\\data_storer\\data_research\\test\\test_GAS_phase_2.csv')
df_processed_test_GAS.rename(columns={'date': 'time'}, inplace=True)
df_processed_test_GAS['time'] = pd.to_datetime(df_processed_test_GAS['time']).dt.strftime('%d/%m/%Y')

df_processed_test_HPG = pd.read_csv('D:\\NCKH\\SARSA_FinancialRL\\data\\data_storer\\data_research\\test\\test_HPG_phase_2.csv')
df_processed_test_HPG.rename(columns={'date': 'time'}, inplace=True)
df_processed_test_HPG['time'] = pd.to_datetime(df_processed_test_HPG['time']).dt.strftime('%d/%m/%Y')

df_processed_test_SSI = pd.read_csv('D:\\NCKH\\SARSA_FinancialRL\\data\\data_storer\\data_research\\test\\test_SSI_phase_2.csv')
df_processed_test_SSI.rename(columns={'date': 'time'}, inplace=True)
df_processed_test_SSI['time'] = pd.to_datetime(df_processed_test_SSI['time']).dt.strftime('%d/%m/%Y')

df_processed_test_VCB = pd.read_csv('D:\\NCKH\\SARSA_FinancialRL\\data\\data_storer\\data_research\\test\\test_VCB_phase_2.csv')
df_processed_test_VCB.rename(columns={'date': 'time'}, inplace=True)
df_processed_test_VCB['time'] = pd.to_datetime(df_processed_test_VCB['time']).dt.strftime('%d/%m/%Y')

# Ghép dữ liệu: train trước rồi đến test (theo thời gian) thay vì dùng toán tử '+'
# df_processed = pd.concat([df_processed_train, df_processed_test], ignore_index=True)
# Đảm bảo thứ tự thời gian tăng dần nếu chưa chắc chắn
# _df_time = pd.to_datetime(df_processed['time'], format='%d/%m/%Y')
# df_processed = df_processed.assign(_time=_df_time).sort_values('_time').drop(columns=['_time']).reset_index(drop=True)
# print(f"✓ Merged train+test: {df_processed.shape} (train={df_processed_train.shape}, test={df_processed_test.shape})")
# print(f"  Time range: {df_processed['time'].iloc[0]} -> {df_processed['time'].iloc[-1]}")


✓ Model loaded: d:\NCKH\SARSA_FinancialRL\models\sarsa_acb_phase_2.pth


In [3]:
# 1.4. Khởi tạo MDP và chạy simulation
mdp_ACB = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_ACB = df_processed_test_ACB.iloc[0]
state_init_ACB = [
    float(first_row_ACB['close']),
    mdp_ACB.balance_init,
    0,
    float(first_row_ACB['MACD']),
    float(first_row_ACB['RSI']),
    float(first_row_ACB['CCI']),
    float(first_row_ACB['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_ACB, rewards_ACB, actions_ACB = mdp_ACB.simulate(
    df_processed_test_ACB[1:].reset_index(drop=True), 
    state_init_ACB, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_ACB)} states, {len(actions_ACB)} actions")
print(f"  Total reward: {sum(rewards_ACB):.2f}")
print(f"  Final portfolio: ${states_ACB[-1][1] + states_ACB[-1][0]*states_ACB[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 250 states, 249 actions
  Total reward: -11.24
  Final portfolio: $988.76


In [4]:
if os.path.exists(model_path_fpt_phase_1):
    state_dict = torch.load(model_path_fpt_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_fpt_phase_1}")
else:
    print(f"✗ Model not found: {model_path_fpt_phase_1}")

✓ Model loaded: d:\NCKH\SARSA_FinancialRL\models\sarsa_fpt_phase_2.pth


In [5]:
# 1.5. Khởi tạo MDP và chạy simulation
mdp_FPT = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_FPT = df_processed_test_FPT.iloc[0]
state_init_FPT = [
    float(first_row_FPT['close']),
    mdp_FPT.balance_init,
    0,
    float(first_row_FPT['MACD']),
    float(first_row_FPT['RSI']),
    float(first_row_FPT['CCI']),
    float(first_row_FPT['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_FPT, rewards_FPT, actions_FPT = mdp_FPT.simulate(
    df_processed_test_FPT[1:].reset_index(drop=True), 
    state_init_FPT, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_FPT)} states, {len(actions_FPT)} actions")
print(f"  Total reward: {sum(rewards_FPT):.2f}")
print(f"  Final portfolio: ${states_FPT[-1][1] + states_FPT[-1][0]*states_FPT[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 250 states, 249 actions
  Total reward: 150.65
  Final portfolio: $1150.65


In [6]:
if os.path.exists(model_path_gas_phase_1):
    state_dict = torch.load(model_path_gas_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_gas_phase_1}")
else:
    print(f"✗ Model not found: {model_path_gas_phase_1}")

✓ Model loaded: d:\NCKH\SARSA_FinancialRL\models\sarsa_gas_phase_2.pth


In [7]:
# 1.4. Khởi tạo MDP và chạy simulation
mdp_GAS = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_GAS = df_processed_test_GAS.iloc[0]
state_init_GAS = [
    float(first_row_GAS['close']),
    mdp_GAS.balance_init,
    0,
    float(first_row_GAS['MACD']),
    float(first_row_GAS['RSI']),
    float(first_row_GAS['CCI']),
    float(first_row_GAS['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_GAS, rewards_GAS, actions_GAS = mdp_GAS.simulate(
    df_processed_test_GAS[1:].reset_index(drop=True), 
    state_init_GAS, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_GAS)} states, {len(actions_GAS)} actions")
print(f"  Total reward: {sum(rewards_GAS):.2f}")
print(f"  Final portfolio: ${states_GAS[-1][1] + states_GAS[-1][0]*states_GAS[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 250 states, 249 actions
  Total reward: 115.27
  Final portfolio: $1115.27


In [8]:
if os.path.exists(model_path_hpg_phase_1):
    state_dict = torch.load(model_path_hpg_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_hpg_phase_1}")
else:
    print(f"✗ Model not found: {model_path_hpg_phase_1}")

✓ Model loaded: d:\NCKH\SARSA_FinancialRL\models\sarsa_hpg_phase_2.pth


In [9]:
# 1.4. Khởi tạo MDP và chạy simulation
mdp_HPG = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_HPG = df_processed_test_HPG.iloc[0]
state_init_HPG = [
    float(first_row_HPG['close']),
    mdp_HPG.balance_init,
    0,
    float(first_row_HPG['MACD']),
    float(first_row_HPG['RSI']),
    float(first_row_HPG['CCI']),
    float(first_row_HPG['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_HPG, rewards_HPG, actions_HPG = mdp_HPG.simulate(
    df_processed_test_HPG[1:].reset_index(drop=True), 
    state_init_HPG, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_HPG)} states, {len(actions_HPG)} actions")
print(f"  Total reward: {sum(rewards_HPG):.2f}")
print(f"  Final portfolio: ${states_HPG[-1][1] + states_HPG[-1][0]*states_HPG[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 250 states, 249 actions
  Total reward: 3.05
  Final portfolio: $1003.05


In [10]:
if os.path.exists(model_path_ssi_phase_1):
    state_dict = torch.load(model_path_ssi_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_ssi_phase_1}")
else:
    print(f"✗ Model not found: {model_path_ssi_phase_1}")

✓ Model loaded: d:\NCKH\SARSA_FinancialRL\models\sarsa_ssi_phase_2.pth


In [11]:
# 1.4. Khởi tạo MDP và chạy simulation
mdp_SSI = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_SSI = df_processed_test_SSI.iloc[0]
state_init_SSI = [
    float(first_row_SSI['close']),
    mdp_SSI.balance_init,
    0,
    float(first_row_SSI['MACD']),
    float(first_row_SSI['RSI']),
    float(first_row_SSI['CCI']),
    float(first_row_SSI['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_SSI, rewards_SSI, actions_SSI = mdp_SSI.simulate(
    df_processed_test_SSI[1:].reset_index(drop=True), 
    state_init_SSI, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_SSI)} states, {len(actions_SSI)} actions")
print(f"  Total reward: {sum(rewards_SSI):.2f}")
print(f"  Final portfolio: ${states_SSI[-1][1] + states_SSI[-1][0]*states_SSI[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 250 states, 249 actions
  Total reward: -355.81
  Final portfolio: $644.19


In [12]:
if os.path.exists(model_path_vcb_phase_1):
    state_dict = torch.load(model_path_vcb_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_vcb_phase_1}")
else:
    print(f"✗ Model not found: {model_path_vcb_phase_1}")

✓ Model loaded: d:\NCKH\SARSA_FinancialRL\models\sarsa_vcb_phase_2.pth


In [13]:
# 1.4. Khởi tạo MDP và chạy simulation
mdp_VCB = StockTradingMDP(balance_init=1000, k=5, min_balance=-100)

def pi_deep(s, eps=0.0, greedy=True):
    with torch.no_grad():
        out_qsa = qsa(torch.Tensor(s).float()).squeeze()
        action = out_qsa.argmax().item() - 5
    return action

# State ban đầu
first_row_VCB = df_processed_test_VCB.iloc[0]
state_init_VCB = [
    float(first_row_VCB['close']),
    mdp_VCB.balance_init,
    0,
    float(first_row_VCB['MACD']),
    float(first_row_VCB['RSI']),
    float(first_row_VCB['CCI']),
    float(first_row_VCB['ADX'])
]

# Chạy simulation
print("\nĐang chạy simulation...")
states_VCB, rewards_VCB, actions_VCB = mdp_VCB.simulate(
    df_processed_test_VCB[1:].reset_index(drop=True), 
    state_init_VCB, 
    pi_deep, 
    greedy=True, 
    eps=0.0
)

print(f"✓ Simulation hoàn tất: {len(states_VCB)} states, {len(actions_VCB)} actions")
print(f"  Total reward: {sum(rewards_VCB):.2f}")
print(f"  Final portfolio: ${states_VCB[-1][1] + states_VCB[-1][0]*states_VCB[-1][2]:.2f}")


Đang chạy simulation...
✓ Simulation hoàn tất: 250 states, 249 actions
  Total reward: 3.93
  Final portfolio: $1003.93


In [14]:
import numpy as np
import torch

def run_simulation_and_decompose_q_balanced(mdp, model, df_test, state_init, w_vector=[1.0, 0.5, 1.0, 0.1], alpha=0.005):
    """
    Phiên bản Cân Bằng (Balanced): 
    Chuẩn hóa Q-total theo % Tài sản để Profit ngang hàng với Risk/Stab.
    """
    # 1. TÍNH TOÁN CÁC CHỈ SỐ KỸ THUẬT (Vectorized)
    prices = df_test['close'].values
    returns = np.zeros_like(prices)
    safe_prices = np.where(prices[:-1] == 0, 1e-9, prices[:-1]) 
    returns[1:] = (prices[1:] - prices[:-1]) / safe_prices
    
    running_max = np.maximum.accumulate(prices)
    MA10 = np.convolve(prices, np.ones(10)/10, mode='same')
    
    # 2. KHỞI TẠO
    states = [state_init]
    actions = []
    rewards = []
    q_value_history = [] 
    
    w1, w2, w3, w4 = w_vector
    current_state = state_init
    
    print(f"Đang chạy simulation (Balanced Mode)...")
    
    # 3. VÒNG LẶP CHÍNH
    for t in range(len(df_test) - 1):
        
        # --- A. LẤY Q-TOTAL TỪ MODEL (Đơn vị: Tiền tích lũy dự kiến) ---
        state_tensor = torch.FloatTensor(current_state)
        with torch.no_grad():
            q_total_scalar = model(state_tensor).squeeze().cpu().numpy()
            
        # --- B. TÍNH TỔNG TÀI SẢN HIỆN TẠI ĐỂ CHUẨN HÓA ---
        # Portfolio = Tiền mặt + (Số cổ phiếu * Giá hiện tại)
        current_portfolio = current_state[1] + (current_state[2] * prices[t])
        
        # Tránh chia cho 0 hoặc số quá nhỏ
        if current_portfolio < 10: current_portfolio = 1000 
            
        # --- C. PHÂN RÃ Q-VALUE (ĐÃ CHUẨN HÓA VỀ %) ---
        q_matrix_t = np.zeros((11, 4))
        
        # Tính toán các chỉ số môi trường
        current_price = prices[t]
        current_dd = (current_price - running_max[t]) / running_max[t] if running_max[t] != 0 else 0
        current_return = returns[t]
        
        trend_val = current_price - MA10[t]
        sign_trend = np.sign(trend_val)
        
        for i in range(11):
            action_val = i - 5 
            
            # --- 1. CHUẨN HÓA Q TỔNG VỀ % (QUAN TRỌNG NHẤT) ---
            # Q ($16.36) / Portfolio ($1000) = 0.01636 (1.6%)
            # Lúc này 0.016 (Profit) đã ngang hàng với 0.025 (Pos)
            q_total_norm = q_total_scalar[i] / current_portfolio
            
            # --- 2. TÍNH CÁC THÀNH PHẦN PHỤ (VỐN ĐÃ LÀ %) ---
            # Risk: % Drawdown
            val_risk = w2 * -abs(current_dd)          
            
            # Stability: % Biến động ngày
            val_stab = w4 * -abs(current_return)      
            
            # Position: % Thưởng Trend
            sign_action = np.sign(action_val)
            if action_val == 0:
                val_pos = 0
            elif sign_action == sign_trend:
                val_pos = w3 * alpha
            else:
                val_pos = w3 * -alpha
                
            # --- 3. TÍNH PROFIT (PHẦN DƯ SAU KHI ĐÃ CHUẨN HÓA) ---
            # Profit (%) = Tổng (%) - (Risk% + Pos% + Stab%)
            val_profit = q_total_norm - (val_risk + val_pos + val_stab)
            
            q_matrix_t[i] = [val_profit, val_risk, val_pos, val_stab]
            
        q_value_history.append(q_matrix_t)
        
        # --- D. CHỌN HÀNH ĐỘNG & BƯỚC ĐI TIẾP THEO ---
        # Lưu ý: Agent vẫn chọn dựa trên Q gốc (Tiền) để đảm bảo đúng logic đã học
        action_idx = np.argmax(q_total_scalar) 
        real_action = action_idx - 5
        
        next_row = df_test.iloc[t + 1]
        next_state, reward, done = mdp.step(current_state, real_action, next_row)
        
        states.append(next_state) 
        actions.append(real_action)
        rewards.append(reward)
        
        current_state = next_state
        if done:
            break
            
    return states, actions, rewards, q_value_history

In [15]:
# 1. Chạy lại Simulation với hàm Balanced
print("Đang tính toán lại Q-values (Chuẩn hóa %)...")
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_ACB, 
    qsa, 
    df_processed_test_ACB, 
    state_init_ACB,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_ACB = states_new
actions_ACB = actions_new
rewards_ACB = rewards_new
q_values_history_ACB = q_values_history_new
# Kỳ vọng: Profit giờ chỉ khoảng 0.016 (1.6%) thay vì 16.3

Đang tính toán lại Q-values (Chuẩn hóa %)...
Đang chạy simulation (Balanced Mode)...


In [16]:
if os.path.exists(model_path_fpt_phase_1):
    state_dict = torch.load(model_path_fpt_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_fpt_phase_1}")
else:
    print(f"✗ Model not found: {model_path_fpt_phase_1}")

✓ Model loaded: d:\NCKH\SARSA_FinancialRL\models\sarsa_fpt_phase_2.pth


In [17]:
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_FPT, 
    qsa, 
    df_processed_test_FPT, 
    state_init_FPT,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_FPT = states_new
actions_FPT = actions_new
rewards_FPT = rewards_new
q_values_history_FPT = q_values_history_new

Đang chạy simulation (Balanced Mode)...


In [18]:
if os.path.exists(model_path_gas_phase_1):
    state_dict = torch.load(model_path_gas_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_gas_phase_1}")
else:
    print(f"✗ Model not found: {model_path_gas_phase_1}")

✓ Model loaded: d:\NCKH\SARSA_FinancialRL\models\sarsa_gas_phase_2.pth


In [19]:
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_GAS, 
    qsa, 
    df_processed_test_GAS, 
    state_init_GAS,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_GAS = states_new
actions_GAS = actions_new
rewards_GAS = rewards_new
q_values_history_GAS = q_values_history_new

Đang chạy simulation (Balanced Mode)...


In [20]:
if os.path.exists(model_path_hpg_phase_1):
    state_dict = torch.load(model_path_hpg_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_hpg_phase_1}")
else:
    print(f"✗ Model not found: {model_path_hpg_phase_1}")

✓ Model loaded: d:\NCKH\SARSA_FinancialRL\models\sarsa_hpg_phase_2.pth


In [21]:
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_HPG, 
    qsa, 
    df_processed_test_HPG, 
    state_init_HPG,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_HPG = states_new
actions_HPG = actions_new
rewards_HPG = rewards_new
q_values_history_HPG = q_values_history_new

Đang chạy simulation (Balanced Mode)...


In [22]:
if os.path.exists(model_path_ssi_phase_1):
    state_dict = torch.load(model_path_ssi_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_ssi_phase_1}")
else:
    print(f"✗ Model not found: {model_path_ssi_phase_1}")

✓ Model loaded: d:\NCKH\SARSA_FinancialRL\models\sarsa_ssi_phase_2.pth


In [23]:
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_SSI, 
    qsa, 
    df_processed_test_SSI, 
    state_init_SSI,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_SSI = states_new
actions_SSI = actions_new
rewards_SSI = rewards_new
q_values_history_SSI = q_values_history_new

Đang chạy simulation (Balanced Mode)...


In [24]:
if os.path.exists(model_path_vcb_phase_1):
    state_dict = torch.load(model_path_vcb_phase_1, map_location=torch.device('cpu'))
    qsa.load_state_dict(state_dict)
    qsa.eval()
    print(f"✓ Model loaded: {model_path_vcb_phase_1}")
else:
    print(f"✗ Model not found: {model_path_vcb_phase_1}")

✓ Model loaded: d:\NCKH\SARSA_FinancialRL\models\sarsa_vcb_phase_2.pth


In [25]:
states_new, actions_new, rewards_new, q_values_history_new = run_simulation_and_decompose_q_balanced(
    mdp_VCB, 
    qsa, 
    df_processed_test_VCB, 
    state_init_VCB,
    w_vector=[0.25, 0.25, 0.25, 0.25], # Giữ nguyên trọng số
    alpha=0.005
)

# 2. Cập nhật lại các biến toàn cục để vẽ biểu đồ
states_VCB = states_new
actions_VCB = actions_new
rewards_VCB = rewards_new
q_values_history_VCB = q_values_history_new

Đang chạy simulation (Balanced Mode)...


In [26]:
import numpy as np

def analyze_msx(q_vector_selected, q_vector_compared, component_names):
    """
    Phân tích Minimal Sufficient Explanation (MSX) cho một cặp hành động.
    
    Args:
        q_vector_selected: Vector Q của hành động được chọn (VD: Mua).
        q_vector_compared: Vector Q của hành động so sánh (VD: Giữ/Bán).
        component_names: List tên các thành phần ['Profit', 'Risk', 'Pos', 'Stab'].
        
    Returns:
        dict: Kết quả phân tích MSX.
    """
    # 1. Tính vector khác biệt (RDX - Reward Difference Explanation) [cite: 1070]
    # Delta > 0: Lý do ủng hộ hành động chọn
    # Delta < 0: Lý do phản đối (ủng hộ hành động kia)
    delta = q_vector_selected - q_vector_compared
    
    # 2. Phân loại Pros (Tích cực) và Cons (Tiêu cực)
    pros = [] 
    disadvantage_sum = 0.0 # Tổng bất lợi (d)
    
    for i, val in enumerate(delta):
        if val > 0:
            pros.append((i, val))
        else:
            disadvantage_sum += abs(val) # [cite: 1077]
            
    # 3. Sắp xếp các lý do tích cực từ lớn đến bé (Greedy approach) [cite: 1082]
    pros.sort(key=lambda x: x[1], reverse=True)
    
    # 4. Tìm tập hợp MSX+ (Minimal Sufficient Set) [cite: 1079]
    msx_plus = []
    current_sum = 0.0
    is_sufficient = False
    
    # Cộng dồn lý do tốt nhất cho đến khi thắng được disadvantage
    for idx, val in pros:
        msx_plus.append({
            'component': component_names[idx],
            'value': val,
            'contribution_percent': 0 # Sẽ tính sau
        })
        current_sum += val
        
        if current_sum > disadvantage_sum:
            is_sufficient = True
            break
            
    # Tính % đóng góp của từng lý do trong tập MSX
    for item in msx_plus:
        item['contribution_percent'] = (item['value'] / current_sum) * 100

    # 5. Xác định các lý do tiêu cực chính (Optional - để hiển thị Disadvantage)
    cons = []
    for i, val in enumerate(delta):
        if val < 0:
            cons.append({'component': component_names[i], 'value': val})

    return {
        'is_dominated': (disadvantage_sum == 0), # Nếu không có bất lợi nào [cite: 1093]
        'msx_plus': msx_plus,
        'disadvantage': disadvantage_sum,
        'cons_details': cons,
        'full_delta': delta
    }

In [27]:
import numpy as np
import pandas as pd
from scipy.signal import argrelextrema

def identify_critical_points(df_test, actions_history, q_values_history=None, action_change_threshold=3, trend_window=10, top_k=15):
    """
    Lọc ra Top K điểm quan trọng nhất từ kết quả chạy Test.
    
    Args:
        top_k (int): Số lượng điểm tối đa muốn lấy (mặc định 15).
    """
    prices = df_test['close'].values
    actions = np.array(actions_history)
    n = len(prices)
    
    # --- GIAI ĐOẠN 1: TÌM TẤT CẢ ỨNG VIÊN (CANDIDATES) ---
    
    # 1. Nhóm Đáy/DD (Priority 1 - Quan trọng nhất)
    idxs_bottom = []
    global_min_idx = np.argmin(prices)
    idxs_bottom.append(global_min_idx)
    
    # Tìm Max Drawdown
    running_max = np.maximum.accumulate(prices)
    # Tránh chia cho 0
    safe_running_max = np.where(running_max == 0, 1e-9, running_max)
    drawdowns = (prices - running_max) / safe_running_max
    max_dd_idx = np.argmin(drawdowns)
    
    if max_dd_idx != global_min_idx:
        idxs_bottom.append(max_dd_idx)
    
    # 2. Nhóm Đảo chiều (Priority 2)
    peaks = argrelextrema(prices, np.greater, order=trend_window)[0]
    valleys = argrelextrema(prices, np.less, order=trend_window)[0]
    idxs_reversal = np.concatenate((peaks, valleys))
    
    # Chấm điểm đảo chiều: Ưu tiên điểm có giá lệch xa nhất so với Median
    median_price = np.median(prices)
    # Tạo list (index, score) và sort giảm dần theo score (độ lệch)
    scored_reversals = [(idx, abs(prices[idx] - median_price)) for idx in idxs_reversal]
    scored_reversals.sort(key=lambda x: x[1], reverse=True)
    sorted_reversal_indices = [x[0] for x in scored_reversals]

    # 3. Nhóm Thay đổi hành động (Priority 3)
    action_diffs = np.abs(actions[1:] - actions[:-1])
    # Lấy index (cộng 1 vì diff làm lệch index)
    shift_candidates = np.where(action_diffs >= action_change_threshold)[0] + 1
    
    # Chấm điểm shift: Ưu tiên cú quay xe gắt nhất (Magnitude lớn nhất)
    scored_shifts = [(idx, action_diffs[idx-1]) for idx in shift_candidates]
    scored_shifts.sort(key=lambda x: x[1], reverse=True)
    sorted_shift_indices = [x[0] for x in scored_shifts]

    # --- GIAI ĐOẠN 2: CHỌN LỌC (FILLING) ---
    final_indices = set()
    
    # Bước 1: Lấy tất cả Đáy/DD
    for idx in idxs_bottom:
        if len(final_indices) < top_k:
            final_indices.add(idx)
            
    # Bước 2: Lấy các điểm Đảo chiều lớn (Điền tiếp vào chỗ trống)
    for idx in sorted_reversal_indices:
        if len(final_indices) < top_k:
            final_indices.add(idx)
            
    # Bước 3: Nếu vẫn chưa đủ, lấy thêm các Action Shift lớn nhất
    for idx in sorted_shift_indices:
        if len(final_indices) < top_k:
            final_indices.add(idx)
            
    # --- GIAI ĐOẠN 3: TỔNG HỢP VÀ PHÂN LOẠI LẠI ---
    sorted_unique_indices = sorted(list(final_indices))
    
    # Mapping lại vào dictionary để biết điểm nào thuộc loại nào
    critical_points = {
        'trend_reversal': [],
        'lowest_price': [],
        'action_shift': []
    }
    
    # Dùng set để lookup cho nhanh
    set_bottoms = set(idxs_bottom)
    set_reversals = set(idxs_reversal)
    set_shifts = set(shift_candidates)
    
    for idx in sorted_unique_indices:
        # Một điểm có thể thuộc nhiều loại
        if idx in set_bottoms:
            critical_points['lowest_price'].append(idx)
        if idx in set_reversals:
            critical_points['trend_reversal'].append(idx)
        if idx in set_shifts:
            critical_points['action_shift'].append(idx)

    print(f"Đã lọc Top {len(sorted_unique_indices)} điểm quan trọng (Limit: {top_k}).")
    print(f"- Đáy/DD sâu: {len(critical_points['lowest_price'])}")
    print(f"- Đảo chiều lớn: {len(critical_points['trend_reversal'])}")
    print(f"- Đổi Action gắt: {len(critical_points['action_shift'])}")
    
    return critical_points, sorted_unique_indices

In [28]:
critical_dict_ACB, index_list_ACB = identify_critical_points(
    df_processed_test_ACB, 
    actions_ACB, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 15 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 1
- Đảo chiều lớn: 14
- Đổi Action gắt: 5


In [29]:
critical_dict_FPT, index_list_FPT = identify_critical_points(
    df_processed_test_FPT, 
    actions_FPT, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 15 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 2
- Đảo chiều lớn: 12
- Đổi Action gắt: 9


In [30]:
critical_dict_GAS, index_list_GAS = identify_critical_points(
    df_processed_test_GAS, 
    actions_GAS, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 15 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 2
- Đảo chiều lớn: 14
- Đổi Action gắt: 3


In [31]:
critical_dict_HPG, index_list_HPG = identify_critical_points(
    df_processed_test_HPG, 
    actions_HPG, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 15 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 1
- Đảo chiều lớn: 15
- Đổi Action gắt: 1


In [32]:
critical_dict_SSI, index_list_SSI = identify_critical_points(
    df_processed_test_SSI, 
    actions_SSI, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 15 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 1
- Đảo chiều lớn: 10
- Đổi Action gắt: 6


In [33]:
critical_dict_VCB, index_list_VCB = identify_critical_points(
    df_processed_test_VCB, 
    actions_VCB, 
    action_change_threshold=3,
    top_k=15 # Chỉ lấy 15 điểm quan trọng nhất
)

Đã lọc Top 15 điểm quan trọng (Limit: 15).
- Đáy/DD sâu: 2
- Đảo chiều lớn: 10
- Đổi Action gắt: 8


In [34]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

for t in index_list_ACB:
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_ACB.iloc[t]['close']
    date = df_processed_test_ACB.iloc[t]['time']
    actual_action = actions_ACB[t]
    actual_reward = rewards_ACB[t] # Lấy reward thực tế
    
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_ACB.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_ACB.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_ACB.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_ACB[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===



>> Thời điểm t=1 (03/01/2019) - Giá: 6.24
>> Loại điểm: [ĐẢO CHIỀU | ĐỔI ACTION]
>> Hành động thực tế: +0
>> Reward thực tế nhận được: 0.000000

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: +0] vs [Bỏ: Mua Mạnh (+5)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.112293 |        0.092389 |       +0.019904
   Risk       |       -0.009259 |       -0.009259 |       +0.000000
   Pos        |        0.000000 |        0.001250 |       -0.001250
   Stab       |       -0.009259 |       -0.009259 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.093774 |        0.075121 |       +0.018654
   
   [GIẢI THÍCH MSX]:
      -> Chọn vì lý do chính: Profit (+0.019904)
      -> Chấp nhận đánh đổ

In [35]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

for t in index_list_FPT:
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_FPT.iloc[t]['close']
    date = df_processed_test_FPT.iloc[t]['time']
    actual_action = actions_FPT[t]
    actual_reward = rewards_FPT[t] # Lấy reward thực tế
    
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_FPT.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_FPT.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_FPT.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_FPT[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===

>> Thời điểm t=1 (03/01/2019) - Giá: 12.88
>> Loại điểm: [ĐẢO CHIỀU | ĐÁY/DD]
>> Hành động thực tế: +2
>> Reward thực tế nhận được: 0.374240

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: +2] vs [Bỏ: Giữ (0)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.021494 |       -0.018637 |       +0.040131
   Risk       |       -0.002498 |       -0.002498 |       +0.000000
   Pos        |        0.001250 |        0.000000 |       +0.001250
   Stab       |       -0.002498 |       -0.002498 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.017748 |       -0.023633 |       +0.041381
   
   [GIẢI THÍCH MSX]:
      -> Quyết định c

In [36]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

# Xác định độ dài an toàn (lấy min của các list dữ liệu)
# Để đảm bảo không bao giờ truy cập quá giới hạn
max_safe_index = min(len(actions_GAS), len(rewards_GAS), len(q_values_history_GAS)) - 1

for t in index_list_GAS:
    # --- SỬA LỖI TẠI ĐÂY: KIỂM TRA INDEX ---
    if t > max_safe_index:
        print(f"⚠️ Cảnh báo: Điểm t={t} vượt quá độ dài dữ liệu Simulation ({max_safe_index}). Bỏ qua.")
        continue
        
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_GAS.iloc[t]['close']
    date = df_processed_test_GAS.iloc[t]['time']
    
    # Truy cập an toàn
    actual_action = actions_GAS[t]
    actual_reward = rewards_GAS[t] 
    q_matrix = q_values_history_GAS[t]
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_GAS.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_GAS.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_GAS.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_GAS[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===

>> Thời điểm t=2 (04/01/2019) - Giá: 50.19
>> Loại điểm: [ĐẢO CHIỀU | ĐÁY/DD]
>> Hành động thực tế: +2
>> Reward thực tế nhận được: 6.319620

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: +2] vs [Bỏ: Giữ (0)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.033331 |        0.014937 |       +0.018394
   Risk       |       -0.006596 |       -0.006596 |       +0.000000
   Pos        |        0.001250 |        0.000000 |       +0.001250
   Stab       |       -0.000596 |       -0.000596 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.027389 |        0.007746 |       +0.019644
   
   [GIẢI THÍCH MSX]:
      -> Quyết định c

In [37]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

for t in index_list_HPG:
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_HPG.iloc[t]['close']
    date = df_processed_test_HPG.iloc[t]['time']
    actual_action = actions_HPG[t]
    actual_reward = rewards_HPG[t] # Lấy reward thực tế
    
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_HPG.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_HPG.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_HPG.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_HPG[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===

>> Thời điểm t=4 (08/01/2019) - Giá: 7.65
>> Loại điểm: [ĐẢO CHIỀU]
>> Hành động thực tế: +4
>> Reward thực tế nhận được: 3.369400

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: +4] vs [Bỏ: Giữ (0)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.126773 |       -0.015772 |       +0.142545
   Risk       |       -0.015912 |       -0.015912 |       +0.000000
   Pos        |        0.001250 |        0.000000 |       +0.001250
   Stab       |       -0.004177 |       -0.004177 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.107934 |       -0.035861 |       +0.143795
   
   [GIẢI THÍCH MSX]:
      -> Quyết định chọn 4 tốt 

In [38]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

# Xác định độ dài an toàn (lấy min của các list dữ liệu)
# Để đảm bảo không bao giờ truy cập quá giới hạn
max_safe_index = min(len(actions_GAS), len(rewards_GAS), len(q_values_history_GAS)) - 1

for t in index_list_SSI:
    # --- SỬA LỖI TẠI ĐÂY: KIỂM TRA INDEX ---
    if t > max_safe_index:
        print(f"⚠️ Cảnh báo: Điểm t={t} vượt quá độ dài dữ liệu Simulation ({max_safe_index}). Bỏ qua.")
        continue
        
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_SSI.iloc[t]['close']
    date = df_processed_test_SSI.iloc[t]['time']
    
    # Truy cập an toàn
    actual_action = actions_SSI[t]
    actual_reward = rewards_SSI[t] 
    q_matrix = q_values_history_SSI[t]
    
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_SSI.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_SSI.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_SSI.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_SSI[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===

>> Thời điểm t=12 (18/01/2019) - Giá: 8.68
>> Loại điểm: [ĐẢO CHIỀU]
>> Hành động thực tế: -2
>> Reward thực tế nhận được: 0.000000

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: -2] vs [Bỏ: Giữ (0)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.078796 |       -0.009379 |       +0.088174
   Risk       |       -0.012582 |       -0.012582 |       +0.000000
   Pos        |        0.001250 |        0.000000 |       +0.001250
   Stab       |       -0.000575 |       -0.000575 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.066889 |       -0.022535 |       +0.089424
   
   [GIẢI THÍCH MSX]:
      -> Quyết định chọn -2 tố

In [39]:
# --- CẤU HÌNH ---
COMPONENTS = ['Profit', 'Risk', 'Pos', 'Stab']

print("\n=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===")

for t in index_list_VCB:
    # 1. Lấy thông tin ngữ cảnh
    price = df_processed_test_VCB.iloc[t]['close']
    date = df_processed_test_VCB.iloc[t]['time']
    actual_action = actions_VCB[t]
    actual_reward = rewards_VCB[t] # Lấy reward thực tế
    
    # Xác định loại điểm quan trọng (để in label)
    labels = []
    if t in critical_dict_VCB.get('trend_reversal', []): labels.append("ĐẢO CHIỀU")
    if t in critical_dict_VCB.get('lowest_price', []): labels.append("ĐÁY/DD")
    if t in critical_dict_VCB.get('action_shift', []): labels.append("ĐỔI ACTION")
    label_str = " | ".join(labels) if labels else "Normal"
    
    print(f"\n{'='*90}")
    print(f">> Thời điểm t={t} ({date}) - Giá: {price:.2f}")
    print(f">> Loại điểm: [{label_str}]")
    print(f">> Hành động thực tế: {actual_action:+}")
    print(f">> Reward thực tế nhận được: {actual_reward:.6f}")
    
    # 2. Lấy Q-Matrix thực tế tại bước t
    q_matrix = q_values_history_VCB[t] # Shape (11, 4)
    idx_selected = actual_action + 5 # Map về index 0-10
    vec_selected = q_matrix[idx_selected]

    # 3. Tạo danh sách các hành động để so sánh (Logic Đối Xứng)
    comparison_candidates = []
    
    if actual_action == 0:
        # Nếu đang Giữ, so sánh với 2 cực trị Mua/Bán mạnh nhất
        comparison_candidates.append(("Mua Mạnh (+5)", 10))
        comparison_candidates.append(("Bán Mạnh (-5)", 0))
    else:
        # Nếu đang Mua/Bán, so sánh với Giữ và Hành động Ngược lại
        opposite_action = -actual_action
        idx_opposite = opposite_action + 5
        
        comparison_candidates.append(("Giữ (0)", 5))
        comparison_candidates.append((f"Ngược lại ({opposite_action:+})", idx_opposite))

    # 4. Vòng lặp so sánh & Hiển thị chi tiết
    for comp_name, comp_idx in comparison_candidates:
        vec_compared = q_matrix[comp_idx]
        
        # --- HIỂN THỊ BẢNG SO SÁNH TRỰC QUAN (RAW VALUES) ---
        print(f"\n   ------------------------------------------------------------------------")
        print(f"   SO SÁNH KỲ VỌNG (Q): [Chọn: {actual_action:+}] vs [Bỏ: {comp_name}]")
        print(f"   ------------------------------------------------------------------------")
        print(f"   {'Component':<10} | {'Selected (Q)':>15} | {'Compared (Q)':>15} | {'Diff (Delta)':>15}")
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # In từng dòng thành phần
        for i, name in enumerate(COMPONENTS):
            val_sel = vec_selected[i]
            val_comp = vec_compared[i]
            diff = val_sel - val_comp
            # Dùng .6f để hiển thị chính xác số nhỏ
            print(f"   {name:<10} | {val_sel:>15.6f} | {val_comp:>15.6f} | {diff:>+15.6f}")
            
        print(f"   {'-'*10} | {'-'*15} | {'-'*15} | {'-'*15}")
        
        # Tính tổng Q để kiểm tra
        total_sel = sum(vec_selected)
        total_comp = sum(vec_compared)
        total_diff = total_sel - total_comp
        print(f"   {'TOTAL Q':<10} | {total_sel:>15.6f} | {total_comp:>15.6f} | {total_diff:>+15.6f}")
        
        # --- CHẠY HÀM MSX LOGIC ---
        # Hàm analyze_msx sẽ tìm tập hợp lý do tối thiểu
        msx_result = analyze_msx(vec_selected, vec_compared, COMPONENTS)
        
        print(f"   \n   [GIẢI THÍCH MSX]:")
        
        if msx_result.get('is_dominated', False):
            print(f"      -> Quyết định chọn {actual_action} tốt hơn hoàn toàn (Dominated).")
        else:
            # Lý do chọn (Pros - Ưu điểm)
            reasons_plus = [f"{x['component']} ({x['value']:+.6f})" for x in msx_result['msx_plus']]
            if reasons_plus:
                print(f"      -> Chọn vì lý do chính: {', '.join(reasons_plus)}")
            else:
                 print(f"      -> (Không tìm thấy lý do vượt trội cụ thể)")

            # Lý do phản đối (Cons - Nhược điểm chấp nhận)
            if 'cons_details' in msx_result and msx_result['cons_details']:
                reasons_minus = [f"{x['component']} ({x['value']:.6f})" for x in msx_result['cons_details']]
                print(f"      -> Chấp nhận đánh đổi: {', '.join(reasons_minus)}")
            elif msx_result['disadvantage'] > 0:
                 print(f"      -> Tổng bất lợi phải vượt qua: {msx_result['disadvantage']:.6f}")

print(f"\n{'='*90}")


=== BẮT ĐẦU PHÂN TÍCH MSX ĐA CHIỀU (CHI TIẾT & TRỰC QUAN) ===

>> Thời điểm t=0 (02/01/2019) - Giá: 22.94
>> Loại điểm: [ĐÁY/DD]
>> Hành động thực tế: -5
>> Reward thực tế nhận được: 0.000000

   ------------------------------------------------------------------------
   SO SÁNH KỲ VỌNG (Q): [Chọn: -5] vs [Bỏ: Giữ (0)]
   ------------------------------------------------------------------------
   Component  |    Selected (Q) |    Compared (Q) |    Diff (Delta)
   ---------- | --------------- | --------------- | ---------------
   Profit     |        0.092991 |        0.090239 |       +0.002751
   Risk       |       -0.000000 |       -0.000000 |       +0.000000
   Pos        |       -0.001250 |        0.000000 |       -0.001250
   Stab       |       -0.000000 |       -0.000000 |       +0.000000
   ---------- | --------------- | --------------- | ---------------
   TOTAL Q    |        0.091741 |        0.090239 |       +0.001501
   
   [GIẢI THÍCH MSX]:
      -> Chọn vì lý do chính: Pro

In [40]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

def plot_agent_reasoning_minimalist(ticker, df_price, actions, q_history, critical_indices, scale=50):
    """
    Phiên bản Minimalist (Final + Auto Normalize + Safe Index Check):
    - Tự động lọc bỏ các index vượt quá giới hạn dữ liệu.
    - Tự động chuẩn hóa Q-value.
    - Mũi tên sát đỉnh cột.
    """
    
    # --- BƯỚC 0a: KIỂM TRA VÀ LỌC INDEX HỢP LỆ (FIX LỖI INDEX ERROR) ---
    # Xác định độ dài an toàn (simulation thường ngắn hơn df gốc 1 đơn vị)
    max_safe_idx = min(len(actions), len(q_history)) - 1
    
    valid_indices = []
    for t in critical_indices:
        if t <= max_safe_idx:
            valid_indices.append(t)
        else:
            # (Optional) In cảnh báo nếu cần thiết
            pass
            
    if len(valid_indices) == 0:
        print(f"⚠️ Không có điểm Critical nào hợp lệ để vẽ cho {ticker}!")
        return

    # --- BƯỚC 0b: CHUẨN HÓA DỮ LIỆU ĐẦU VÀO ---
    # Gom tất cả các ma trận Q tại các điểm HỢP LỆ lại để tìm max
    all_critical_q = []
    for t in valid_indices:
        all_critical_q.append(q_history[t])
    
    all_critical_q = np.array(all_critical_q)
    max_abs_q = np.max(np.abs(all_critical_q))
    if max_abs_q == 0: max_abs_q = 1 
    
    print(f"Max Abs Q ban đầu: {max_abs_q:.4f}. Đang chuẩn hóa...")

    # --- 1. CHUẨN BỊ DỮ LIỆU ---
    # ... (Khởi tạo các list chứa dữ liệu) ...
    buy_x, buy_y = [], []
    sell_x, sell_y = [], []
    hold_x, hold_y = [], []
    
    x_dates_hidden = [] 
    x_actions = []      
    
    y_profit, y_risk, y_pos, y_stab = [], [], [], []
    
    selected_marker_x_grp = []
    selected_marker_x_act = []
    selected_marker_y = []
    
    tick_vals_top = []
    tick_text_top = []
    
    # Duyệt qua danh sách VALID INDICES thay vì critical_indices gốc
    for i, t in enumerate(valid_indices):
        date_str = df_price.iloc[t]['time']
        price = df_price.iloc[t]['close']
        act = actions[t]
        
        # --- Data Subplot 1 ---
        tick_vals_top.append(date_str)
        tick_text_top.append(date_str) 
        
        if act > 0:
            buy_x.append(date_str); buy_y.append(price)
        elif act < 0:
            sell_x.append(date_str); sell_y.append(price)
        else:
            hold_x.append(date_str); hold_y.append(price)
            
        # --- Data Subplot 2 ---
        q_mat_raw = q_history[t]
        # Chuẩn hóa về [-1, 1] rồi nhân scale
        q_mat = (q_mat_raw / max_abs_q) * scale
        
        group_id = f"#{i+1}" 
        
        compare_set = [
            ("Sell (-5)", 0),
            ("Hold (0)", 5),
            ("Buy (+5)", 10)
        ]
        
        for label, idx in compare_set:
            vec = q_mat[idx]
            
            x_dates_hidden.append(group_id) 
            x_actions.append(label)
            
            y_profit.append(vec[0])
            y_risk.append(vec[1])
            y_pos.append(vec[2])
            y_stab.append(vec[3])
            
            is_selected = False
            if act < 0 and idx == 0: is_selected = True
            elif act == 0 and idx == 5: is_selected = True
            elif act > 0 and idx == 10: is_selected = True
            
            if is_selected:
                selected_marker_x_grp.append(group_id)
                selected_marker_x_act.append(label)
                
                # Tính vị trí sát cột (Margin 2%)
                total_pos = sum([v for v in vec if v > 0])
                if total_pos > 0:
                    marker_y = total_pos + (scale * 0.02)
                else:
                    marker_y = 0 + (scale * 0.02)
                    
                selected_marker_y.append(marker_y)

    # --- 2. VẼ BIỂU ĐỒ ---
    fig = make_subplots(
        rows=2, cols=1, 
        vertical_spacing=0.1,
        row_heights=[0.6, 0.4],
        specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
    )

    # === SUBPLOT 1: PRICE ===
    fig.add_trace(go.Scatter(x=df_price['time'], y=df_price['close'], mode='lines', name='Price', line=dict(color='black', width=1), hoverinfo='x+y'), row=1, col=1)
    
    # Markers
    fig.add_trace(go.Scatter(x=buy_x, y=buy_y, mode='markers', name='Buy', marker=dict(symbol='triangle-up', size=14, color='green', line=dict(width=1, color='black'))), row=1, col=1)
    fig.add_trace(go.Scatter(x=sell_x, y=sell_y, mode='markers', name='Sell', marker=dict(symbol='triangle-down', size=14, color='red', line=dict(width=1, color='black'))), row=1, col=1)
    fig.add_trace(go.Scatter(x=hold_x, y=hold_y, mode='markers', name='Hold', marker=dict(symbol='circle', size=10, color='#f1c40f', line=dict(width=1, color='black'))), row=1, col=1)

    # === SUBPLOT 2: STACKED BARS ===
    for name, y_data, color in zip(
        ['Profit', 'Risk', 'Trend', 'Stability'],
        [y_profit, y_risk, y_pos, y_stab],
        ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']
    ):
        fig.add_trace(go.Bar(
            x=[x_dates_hidden, x_actions], 
            y=y_data, name=name, marker_color=color, legendgroup='comp'
        ), row=2, col=1)

    # Marker Selected (Mũi tên có Legend)
    fig.add_trace(go.Scatter(
        x=[selected_marker_x_grp, selected_marker_x_act], 
        y=selected_marker_y,
        mode='markers',
        marker=dict(symbol='triangle-down', size=14, color='black'),
        name='Agent Selection',
        showlegend=True
    ), row=2, col=1)

    # --- 3. LAYOUT TINH CHỈNH ---
    fig.update_layout(
        height=900,
        plot_bgcolor='white',
        barmode='relative',
        bargap=0.4,       
        bargroupgap=0.05,
        title_text=f"Phân tích Quyết định Agent: {ticker}",
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    # TINH CHỈNH TRỤC X1 (TRÊN)
    fig.update_xaxes(
        tickmode='array',
        tickvals=tick_vals_top,
        ticktext=tick_text_top,
        tickangle=-90, 
        showgrid=True, gridcolor='#eee',
        row=1, col=1
    )
    
    # TINH CHỈNH TRỤC X2 (DƯỚI)
    fig.update_xaxes(
        type='multicategory',
        tickangle=-90, 
        showticklabels=True,
        title_text="", 
        row=2, col=1
    )
    
    # Các trục Y
    fig.update_yaxes(title_text="Giá (Price)", showgrid=True, gridcolor='#eee', row=1, col=1)
    fig.update_yaxes(title_text=f"Norm. Q-Value (x{scale})", showgrid=True, gridcolor='#eee', row=2, col=1)
    
    fig.add_hline(y=0, line_dash="solid", line_color="black", line_width=1, row=2, col=1)
    fig.update_xaxes(rangeslider_visible=False, row=1, col=1)

    return fig # Trả về fig để hiển thị

# --- CHẠY THỬ ---
# fig = plot_agent_reasoning_minimalist(
#     ticker='GAS',
#     df_price=df_processed_test_GAS,
#     actions=actions_GAS,
#     q_history=q_values_history_GAS,
#     critical_indices=index_list_GAS,
#     scale=50
# )
# fig.show()

In [41]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
plot_agent_reasoning_minimalist(
    ticker='ACB',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_ACB,  # Dữ liệu giá
    actions=actions_ACB,             # Hành động thực tế
    q_history=q_values_history_ACB,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_ACB,     # Danh sách các điểm cần vẽ
    scale=10                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)

Max Abs Q ban đầu: 0.1872. Đang chuẩn hóa...


In [42]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
plot_agent_reasoning_minimalist(
    ticker='FPT',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_FPT,  # Dữ liệu giá
    actions=actions_FPT,             # Hành động thực tế
    q_history=q_values_history_FPT,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_FPT,     # Danh sách các điểm cần vẽ
    scale=10                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)

Max Abs Q ban đầu: 0.1301. Đang chuẩn hóa...


In [43]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
plot_agent_reasoning_minimalist(
    ticker='GAS',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_GAS,  # Dữ liệu giá
    actions=actions_GAS,             # Hành động thực tế
    q_history=q_values_history_GAS,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_GAS,     # Danh sách các điểm cần vẽ
    scale=10                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)

Max Abs Q ban đầu: 0.0796. Đang chuẩn hóa...


In [44]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
plot_agent_reasoning_minimalist(
    ticker='HPG',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_HPG,  # Dữ liệu giá
    actions=actions_HPG,             # Hành động thực tế
    q_history=q_values_history_HPG,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_HPG,     # Danh sách các điểm cần vẽ
    scale=10                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)

Max Abs Q ban đầu: 0.1268. Đang chuẩn hóa...


In [45]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
plot_agent_reasoning_minimalist(
    ticker='SSI',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_SSI,  # Dữ liệu giá
    actions=actions_SSI,             # Hành động thực tế
    q_history=q_values_history_SSI,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_SSI,     # Danh sách các điểm cần vẽ
    scale=10                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)

Max Abs Q ban đầu: 0.1108. Đang chuẩn hóa...


In [46]:
# 2. Gọi hàm vẽ (Truyền đúng tên biến vào)
plot_agent_reasoning_minimalist(
    ticker='VCB',                    # Tên mã cổ phiếu hiển thị trên tiêu đề
    df_price=df_processed_test_VCB,  # Dữ liệu giá
    actions=actions_VCB,             # Hành động thực tế
    q_history=q_values_history_VCB,      # Lịch sử Q-value (Input quan trọng)
    critical_indices=index_list_VCB,     # Danh sách các điểm cần vẽ
    scale=10                       # Thang đo hiển thị (để cột cao khoảng 50 đơn vị cho đẹp)
)

Max Abs Q ban đầu: 0.1922. Đang chuẩn hóa...


In [47]:
import ipywidgets as widgets
from IPython.display import display

# 1. Cấu hình dữ liệu cho 6 Agent
# Đảm bảo bạn đã load đủ dữ liệu cho các biến này trước đó
agents_data = {
    'ACB': (df_processed_test_ACB, actions_ACB, q_values_history_ACB, index_list_ACB),
    'FPT': (df_processed_test_FPT, actions_FPT, q_values_history_FPT, index_list_FPT),
    'GAS': (df_processed_test_GAS, actions_GAS, q_values_history_GAS, index_list_GAS),
    'HPG': (df_processed_test_HPG, actions_HPG, q_values_history_HPG, index_list_HPG),
    'SSI': (df_processed_test_SSI, actions_SSI, q_values_history_SSI, index_list_SSI),
    'VCB': (df_processed_test_VCB, actions_VCB, q_values_history_VCB, index_list_VCB),
}

# 2. Tạo danh sách các Figure
figures = {}
print("Đang khởi tạo các biểu đồ...")

for ticker, (df, acts, q_hist, idxs) in agents_data.items():
    # Gọi hàm (nhớ là hàm này phải return fig nhé)
    fig = plot_agent_reasoning_minimalist(
        ticker=ticker,
        df_price=df,
        actions=acts,
        q_history=q_hist,
        critical_indices=idxs,
        scale=50 # Hoặc 10 tùy bạn chỉnh
    )
    figures[ticker] = fig
    print(f"- Đã vẽ xong {ticker}")

print("Hoàn tất!")

Đang khởi tạo các biểu đồ...
Max Abs Q ban đầu: 0.1872. Đang chuẩn hóa...
- Đã vẽ xong ACB
Max Abs Q ban đầu: 0.1301. Đang chuẩn hóa...
- Đã vẽ xong FPT
Max Abs Q ban đầu: 0.0796. Đang chuẩn hóa...
- Đã vẽ xong GAS
Max Abs Q ban đầu: 0.1268. Đang chuẩn hóa...
- Đã vẽ xong HPG
Max Abs Q ban đầu: 0.1108. Đang chuẩn hóa...
- Đã vẽ xong SSI
Max Abs Q ban đầu: 0.1922. Đang chuẩn hóa...
- Đã vẽ xong VCB
Hoàn tất!


In [48]:
# Tạo danh sách các Widget Output (mỗi tab là một Output)
outputs = []
titles = []

for ticker, fig in figures.items():
    out = widgets.Output()
    with out:
        # Hiển thị biểu đồ trong container này
        fig.show()
    outputs.append(out)
    titles.append(ticker)

# Tạo Widget Tab
tabs = widgets.Tab(children=outputs)

# Đặt tên cho từng Tab
for i, title in enumerate(titles):
    tabs.set_title(i, title)

# Hiển thị giao diện tổng hợp
print("\n=== BẢNG ĐIỀU KHIỂN TỔNG HỢP (NHẤN VÀO TAB ĐỂ XEM) ===")
display(tabs)


=== BẢNG ĐIỀU KHIỂN TỔNG HỢP (NHẤN VÀO TAB ĐỂ XEM) ===


In [49]:
# Gỡ bản hiện tại và cài bản 0.1.0post1
!pip uninstall -y kaleido
!pip install "kaleido==0.1.0post1"

  Using cached kaleido-0.1.0.post1-py2.py3-none-win_amd64.whl.metadata (15 kB)
Using cached kaleido-0.1.0.post1-py2.py3-none-win_amd64.whl (56.0 MB)


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\Admin\\anaconda3\\Lib\\site-packages\\kaleido\\executable\\version'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

def plot_all_agents_combined(agents_data, scale=50):
    """
    Vẽ 6 Agent trong 1 khung hình lớn (Grid 3x2).
    Mỗi ô lưới chứa 2 biểu đồ con (Giá trên, Q-Bar dưới).
    """
    
    # Danh sách các mã (Keys)
    tickers = list(agents_data.keys())
    num_agents = len(tickers)
    
    # Cấu hình lưới: 6 hàng (3 hàng agent * 2 subplots/agent), 2 cột
    # row_heights: Phân bổ chiều cao cho Giá (lớn hơn) và Bar (nhỏ hơn)
    # Pattern: [Giá, Bar, Giá, Bar, Giá, Bar]
    row_specs = [0.20, 0.13] * 3 # Tổng xấp xỉ 1.0
    
    # Tiêu đề cho từng Agent (chỉ hiện ở biểu đồ giá)
    subplot_titles = []
    for i in range(0, 6, 2): # Duyệt theo cặp agent: (0,1), (2,3), (4,5)
        # Hàng hiện tại có 2 agent (trái và phải)
        idx_left = i 
        idx_right = i + 1
        
        title_left = f"<b>Agent: {tickers[idx_left]}</b>" if idx_left < num_agents else ""
        title_right = f"<b>Agent: {tickers[idx_right]}</b>" if idx_right < num_agents else ""
        
        # Thêm tiêu đề cho hàng Price
        subplot_titles.extend([title_left, title_right])
        # Hàng Bar không cần tiêu đề
        subplot_titles.extend(["", ""])

    # Tạo Figure lớn
    fig = make_subplots(
        rows=6, cols=2,
        row_heights=[0.6, 0.4, 0.6, 0.4, 0.6, 0.4], # Tỉ lệ chiều cao Price vs Bar
        vertical_spacing=0.06,
        horizontal_spacing=0.05,
        subplot_titles=subplot_titles,
        specs=[[{"secondary_y": False}, {"secondary_y": False}]] * 6
    )

    print("Đang tổng hợp dữ liệu và vẽ...")

    # --- VÒNG LẶP QUA TỪNG AGENT ---
    for i, ticker in enumerate(tickers):
        # Lấy dữ liệu
        df_price, actions, q_history, critical_indices = agents_data[ticker]
        
        # === QUAN TRỌNG: Lọc các index hợp lệ ===
        # Đảm bảo các index không vượt quá độ dài của dữ liệu
        max_idx = min(len(df_price), len(actions), len(q_history)) - 1
        critical_indices = [idx for idx in critical_indices if 0 <= idx <= max_idx]
        
        if len(critical_indices) == 0:
            print(f"⚠️ Cảnh báo: Agent {ticker} không có critical points hợp lệ!")
            continue
        
        # Xác định vị trí trong lưới (Row, Col)
        # i = 0 -> Row 1,2 Col 1
        # i = 1 -> Row 1,2 Col 2
        # i = 2 -> Row 3,4 Col 1
        col_idx = (i % 2) + 1
        base_row = (i // 2) * 2 + 1 # Row bắt đầu của Agent (1, 3, 5)
        row_price = base_row
        row_bar = base_row + 1
        
        # --- XỬ LÝ DỮ LIỆU (Copy từ hàm Minimalist) ---
        all_critical_q = np.array([q_history[t] for t in critical_indices])
        max_abs_q = np.max(np.abs(all_critical_q)) if len(all_critical_q) > 0 else 1
        if max_abs_q == 0: max_abs_q = 1

        buy_x, buy_y = [], []
        sell_x, sell_y = [], []
        hold_x, hold_y = [], []
        
        x_dates_hidden, x_actions = [], []
        y_profit, y_risk, y_pos, y_stab = [], [], [], []
        
        sel_grp, sel_act, sel_y = [], [], []
        tick_vals, tick_text = [], []

        for j, t in enumerate(critical_indices, start=1):  # Bắt đầu đếm từ 1
            date_str = df_price.iloc[t]['time']
            price = df_price.iloc[t]['close']
            act = actions[t]
            
            tick_vals.append(date_str)
            tick_text.append(date_str)
            
            if act > 0: buy_x.append(date_str); buy_y.append(price)
            elif act < 0: sell_x.append(date_str); sell_y.append(price)
            else: hold_x.append(date_str); hold_y.append(price)
            
            q_mat = (q_history[t] / max_abs_q) * scale
            grp_id = f"{j}" # ID nhóm bắt đầu từ 1
            
            compare_set = [("Sell", 0), ("Hold", 5), ("Buy", 10)]
            
            for label, idx in compare_set:
                vec = q_mat[idx]
                x_dates_hidden.append(grp_id)
                x_actions.append(label)
                y_profit.append(vec[0]); y_risk.append(vec[1])
                y_pos.append(vec[2]); y_stab.append(vec[3])
                
                is_selected = False
                if (act < 0 and idx == 0) or (act == 0 and idx == 5) or (act > 0 and idx == 10):
                    is_selected = True
                
                if is_selected:
                    sel_grp.append(grp_id)
                    sel_act.append(label)
                    pos_h = sum([v for v in vec if v > 0])
                    sel_y.append((pos_h if pos_h > 0 else 0) + scale * 0.05)

        # --- VẼ LÊN SUBPLOTS ---
        # Chỉ hiện legend cho agent đầu tiên
        show_legend = (i == 0)
        
        # 1. PRICE CHART
        fig.add_trace(go.Scatter(x=df_price['time'], y=df_price['close'], mode='lines', line=dict(color='black', width=1), showlegend=False), row=row_price, col=col_idx)
        fig.add_trace(go.Scatter(x=buy_x, y=buy_y, mode='markers', marker=dict(symbol='triangle-up', size=10, color='green'), name='Buy', showlegend=show_legend, legendgroup='act'), row=row_price, col=col_idx)
        fig.add_trace(go.Scatter(x=sell_x, y=sell_y, mode='markers', marker=dict(symbol='triangle-down', size=10, color='red'), name='Sell', showlegend=show_legend, legendgroup='act'), row=row_price, col=col_idx)
        fig.add_trace(go.Scatter(x=hold_x, y=hold_y, mode='markers', marker=dict(symbol='circle', size=8, color='#f1c40f'), name='Hold', showlegend=show_legend, legendgroup='act'), row=row_price, col=col_idx)

        # 2. BAR CHART
        for name, y_d, color in zip(['Profit', 'Risk', 'Trend', 'Stability'], [y_profit, y_risk, y_pos, y_stab], ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']):
            fig.add_trace(go.Bar(x=[x_dates_hidden, x_actions], y=y_d, name=name, marker_color=color, showlegend=show_legend, legendgroup='comp'), row=row_bar, col=col_idx)
            
        # Marker Selection
        fig.add_trace(go.Scatter(x=[sel_grp, sel_act], y=sel_y, mode='markers', marker=dict(symbol='triangle-down', size=10, color='black'), name='Agent Choice', showlegend=show_legend, legendgroup='choice'), row=row_bar, col=col_idx)

        # --- FORMAT TRỤC CHO TỪNG Ô ---
        # Trục X Price: Chỉ hiện tick tại ngày critical
        fig.update_xaxes(tickmode='array', tickvals=tick_vals, ticktext=['']*len(tick_vals), showgrid=True, row=row_price, col=col_idx) # Ẩn text ngày ở biểu đồ trên cho đỡ rối
        
        # Trục X Bar: Hiện ngày + hành động
        # Mẹo: Để hiển thị ngày tháng ở trục dưới cùng, ta dùng title hoặc ticktext tùy biến
        fig.update_xaxes(
            type='multicategory', 
            tickangle=-90, 
            showticklabels=True, 
            title_text=f"Critical Points ({len(critical_indices)})",
            title_font=dict(size=10),
            row=row_bar, col=col_idx
        )
        
        # Trục Y
        fig.update_yaxes(showgrid=True, gridcolor='#eee', row=row_price, col=col_idx)
        fig.update_yaxes(showgrid=True, gridcolor='#eee', row=row_bar, col=col_idx)
        
        # Đường 0
        fig.add_hline(y=0, line_color="black", line_width=1, row=row_bar, col=col_idx)

    # --- 3. LAYOUT CHUNG ---
    fig.update_layout(
        height=2000, 
        width=1900,  
        plot_bgcolor='white',
        barmode='relative',
        bargap=0.3, bargroupgap=0.05,
        # title_text="TỔNG HỢP PHÂN TÍCH QUYẾT ĐỊNH CỦA 6 AGENT (ACROSS-MARKET ANALYSIS)",
        title_font=dict(size=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.005, xanchor="center", x=0.5, bgcolor='rgba(255,255,255,0.8)'),
        margin=dict(t=100, b=50, l=50, r=50)
    )

    # SỬA ĐOẠN CUỐI CÙNG NÀY:
    # Thay vì chỉ fig.show(), hãy trả về fig
    return fig 

# --- LỆNH CHẠY VÀ LƯU ẢNH ---

# 1. Tạo đối tượng biểu đồ
fig_final = plot_all_agents_combined(agents_data, scale=50)

# 2. Hiển thị (để xem trước)
fig_final.show()

# 3. Lưu thành file PNG
print("Đang xuất file ảnh PNG (có thể mất vài giây)...")
fig_final.write_image("Tong_Hop_6_Agents_Full_phase_2.png", width=1900, height=2000, scale=1)
print("✅ Đã lưu xong: Tong_Hop_6_Agents_Full_phase_2.png")

Đang tổng hợp dữ liệu và vẽ...


Đang xuất file ảnh PNG (có thể mất vài giây)...
✅ Đã lưu xong: Tong_Hop_6_Agents_Full_phase_2.png
